# **Pipeline Walkthrough**

In [2]:
#@title 0. Environment Setup

from pathlib import Path
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')

print("\nInstalling system dependencies...")
!apt-get update -qq
!apt-get install -y graphviz plantuml -qq

print("\nLocating requirements.txt...")

req_path = None

for p in Path(".").rglob("requirements.txt"):
    req_path = p
    break

if req_path is None:
    raise FileNotFoundError("requirements.txt not found in current directory tree.")

print(f"Found requirements file at: {req_path}")

print("\nInstalling Python dependencies...")
!pip install -q -r "{req_path}"

print("\nDownloading required dataset (Google News Word2Vec)...")

try:
    import kagglehub
    path = kagglehub.dataset_download("leadbest/googlenewsvectorsnegative300")
    print("Dataset downloaded to:", path)

except Exception as e:
    print("Dataset download failed.")
    print("Check Kaggle API configuration.")
    print("Error:", e)

print("\nEnvironment ready.")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Installing system dependencies...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

Locating requirements.txt...
Found requirements file at: drive/MyDrive/hybridvision/requirements.txt

Installing Python dependencies...
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [15]:
#@title 1. Load project and prepare configuration

from pathlib import Path
from datetime import datetime
import sys
import yaml
from google.colab import drive
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)

# ---------------------------------------------------------------------
# 1. Mount Google Drive
# ---------------------------------------------------------------------
print("Mounting Google Drive...")
drive.mount("/content/drive")

# ---------------------------------------------------------------------
# 2. Define project root
# ---------------------------------------------------------------------
project_name = "multimodal-segmentation-pipeline"
project_root = Path("/content/drive/MyDrive") / project_name

if not project_root.exists():
    raise FileNotFoundError(
        f"Project folder not found at: {project_root}"
    )

print(f"Project root found at: {project_root}")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# ---------------------------------------------------------------------
# 3. Load base configuration
# ---------------------------------------------------------------------
config_path = project_root / "configs" / "config.yaml"

if not config_path.exists():
    raise FileNotFoundError(
        f"Configuration file not found at: {config_path}"
    )

print(f"Loading configuration from: {config_path}")

with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# ---------------------------------------------------------------------
# 4. Define essential paths for this demo run
# ---------------------------------------------------------------------
image_dir = project_root / "datasets" / "images" / "single_test"
words_json_path = project_root / "datasets" / "words.json"

if not image_dir.exists():
    raise FileNotFoundError(
        f"Image directory not found at: {image_dir}"
    )

if not words_json_path.exists():
    raise FileNotFoundError(
        f"Words JSON file not found at: {words_json_path}"
    )

config["image_dir"] = str(image_dir)
config["words_json_path"] = str(words_json_path)

if "labeler" in config and isinstance(config["labeler"], dict):
    config["labeler"]["words_json_path"] = str(words_json_path)

# ---------------------------------------------------------------------
# 5. Create a timestamped output directory
# ---------------------------------------------------------------------
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_name = f"run_single_image_test_{timestamp}"
run_output_dir = project_root / "output" / run_name
run_output_dir.mkdir(parents=True, exist_ok=True)

config["output_dir"] = str(run_output_dir)

# ---------------------------------------------------------------------
# 6. Save the effective configuration used in this run
# ---------------------------------------------------------------------
config_save_path = run_output_dir / "config_used.yaml"

with open(config_save_path, "w", encoding="utf-8") as f:
    yaml.dump(config, f, sort_keys=False)

# ---------------------------------------------------------------------
# 7. Summary
# ---------------------------------------------------------------------
print("\nConfiguration prepared successfully.")
print(f"Image directory:   {config['image_dir']}")
print(f"Words JSON path:   {config['labeler']['words_json_path'] if 'labeler' in config else config['words_json_path']}")
print(f"Output directory:  {config['output_dir']}")
print(f"Saved config file: {config_save_path}")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root found at: /content/drive/MyDrive/multimodal-segmentation-pipeline
Loading configuration from: /content/drive/MyDrive/multimodal-segmentation-pipeline/configs/config.yaml

Configuration prepared successfully.
Image directory:   /content/drive/MyDrive/multimodal-segmentation-pipeline/datasets/images/single_test
Words JSON path:   /content/drive/MyDrive/multimodal-segmentation-pipeline/datasets/words.json
Output directory:  /content/drive/MyDrive/multimodal-segmentation-pipeline/output/run_single_image_test_2026-03-23_19-33-01
Saved config file: /content/drive/MyDrive/multimodal-segmentation-pipeline/output/run_single_image_test_2026-03-23_19-33-01/config_used.yaml


In [17]:
#@title 2. Run pipeline

from src.pipeline import Pipeline

pipeline = Pipeline(config)
pipeline_results = pipeline.run()

print("\nPipeline execution finished successfully.")
print("\nReturned results dictionary:")
print(pipeline_results)

2026-03-23 19:34:12,335 - INFO - Initializing architectural components...
2026-03-23 19:34:12,336 - INFO - Preprocessor initialized with config: {'use_clahe': False, 'use_gaussian_blur': False, 'use_grayscale': False}
2026-03-23 19:34:12,338 - INFO - Initializing SAM 2 Wrapper by loading 'facebook/sam2-hiera-large' from Hugging Face Hub...
2026-03-23 19:34:12,479 - INFO - HTTP Request: HEAD https://huggingface.co/facebook/sam2-hiera-large/resolve/main/sam2_hiera_large.pt "HTTP/1.1 302 Found"


Initializing pipeline...


2026-03-23 19:34:14,741 - INFO - Loaded checkpoint sucessfully
2026-03-23 19:34:15,054 - INFO - SAM 2 Wrapper Initialized successfully from Hugging Face.
2026-03-23 19:34:15,055 - INFO - FinalValidator initialized with weights: {'clip_confidence': 1, 'dino_iou': 0, 'resnet_iou': 0}
2026-03-23 19:34:15,059 - INFO - --- Starting Main Pipeline Run ---
2026-03-23 19:34:15,063 - INFO - --- Phase 2: K-Value Estimation and SAM Mask Generation (SAM2-first) ---
2026-03-23 19:34:15,064 - INFO - Processing k-values and masks for image 000000002149.jpg...
2026-03-23 19:34:15,082 - INFO - Loaded <class 'dict'> from k_estimates_8352795db8abb01f4214471821bc4719_-4878091700395527247.pkl.
2026-03-23 19:34:15,083 - INFO - Loaded k_semantic (10) and k_structural (7) from cache.
2026-03-23 19:34:15,083 - INFO - Processing k-values and masks for image 000000014007.jpg...
2026-03-23 19:34:15,105 - INFO - Loaded <class 'dict'> from k_estimates_a77934f62308e3a57de97621e9d9e3b3_-4878091700395527247.pkl.
2026-0

Running pipeline...


2026-03-23 19:34:15,655 - INFO - Loading CLIP model: ViT-B/32...
2026-03-23 19:34:19,031 - INFO - Hook successfully registered on layer: visual.transformer.resblocks.11
2026-03-23 19:34:22,975 - INFO - Encoding 80 words with 11 templates each...
2026-03-23 19:34:25,056 - INFO - --- Phase 1: Global Feature Extraction ---
2026-03-23 19:34:25,060 - INFO - Determined semantic k-values: []
2026-03-23 19:34:25,062 - INFO - Determined structural k-values: []
2026-03-23 19:34:25,062 - INFO - --- Phase 3: Multi-View Analysis ---
2026-03-23 19:34:25,063 - INFO - --- Phase 4 & 5: Running Final Validation and Output ---
2026-03-23 19:34:25,069 - INFO - Loaded <class 'tuple'> from clip_features_for_validation_ViT-B-32_transformer.resblocks.11.pkl.
2026-03-23 19:34:25,070 - INFO - Loading cached CLIP features from /content/drive/MyDrive/multimodal-segmentation-pipeline/output/run_single_image_test_2026-03-23_19-33-01/features_cache/clip_features_for_validation_ViT-B-32_transformer.resblocks.11.pkl
2


Pipeline execution finished successfully.

Returned results dictionary:
({'000000002149.jpg': [{'mask': array([[ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False],
       [ True,  True,  True, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]]), 'label': 'unknown', 'clip_info': {'best_label': 'apple', 'best_similarity': 0.26733580231666565, 'margin': 0.016467750072479248, 'top_matches': [{'label': 'apple', 'similarity': 0.26733580231666565}, {'label': 'bus', 'similarity': 0.2508680522441864}, {'label': 'handbag', 'similarity': 0.24709181487560272}, {'label': 'laptop', 'similarity': 0.24659138917922974}, {'label': 'refrigerator', 'similarity': 0.24445487558841705}]}, 'final_confidence': 0.26733580231666565, 'status': 'VALIDATED_CLIP_ONLY', 'scores_breakdown': {'dino_cluster_id': -1